In [1]:
print("Jupyter is on.")

Jupyter is on.


In [2]:
!pip install -U transformers accelerate pillow pandas tqdm
!pip install -U bitsandbytes

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------- ----- 8.4/9.7 MB 48.7 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 35.6 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.0 which is incompatible.


In [3]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


LlavaForConditionalGeneration(
  (model): LlavaModel(
    (vision_tower): CLIPVisionModel(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
       

In [13]:
from PIL import Image
import torch

def resize_for_vlm(img: Image.Image, max_side: int = 768) -> Image.Image:
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    img = img.copy()
    img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    return img


def llava_describe(image: Image.Image, max_new_tokens: int = 160) -> str:
    instruction = (
        "Describe this artwork image in detail. "
        "Focus on composition, visual structure, mood, style, symbolism, "
        "and what kind of meaning or idea the image seems to express. "
        "Base everything strictly on what is visible."
    )

    # IMPORTANT: include the image placeholder token
    prompt = "<image>\n" + instruction

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items() if torch.is_tensor(v)}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    text = processor.batch_decode(out, skip_special_tokens=True)[0].strip()

    # remove prompt echo if present
    if instruction in text:
        text = text.split(instruction, 1)[-1].strip()

    return text



In [14]:
import os
import time
from tqdm import tqdm

import pandas as pd
from PIL import Image

images_dir = r"./images/sample"
out_csv = r"./out/llava_captions.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)

img_exts = (".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff", ".bmp")

if not os.path.isdir(images_dir):
    raise FileNotFoundError(f"images_dir not found: {images_dir}")

# checkpoint: considerar "done" apenas o que já foi OK
done = set()
if os.path.exists(out_csv):
    try:
        df_done = pd.read_csv(out_csv)
        if "status" in df_done.columns:
            df_done = df_done[df_done["status"].astype(str) == "ok"]
        done = set(df_done["image_filename"].astype(str).tolist())
    except Exception:
        done = set()

files = sorted([fn for fn in os.listdir(images_dir) if fn.lower().endswith(img_exts)])
todo = [fn for fn in files if fn not in done]
print("total images:", len(files), "already done (ok):", len(done), "to do:", len(todo))

rows_buffer = []
flush_every = 10

for fn in tqdm(todo, total=len(todo), mininterval=1.0):

    try:
        img = Image.open(path).convert("RGB")
        img = resize_for_vlm(img, max_side=768)
        print("starting:", fn)
        desc = llava_describe(img, max_new_tokens=40)
        print("done:", fn, "chars:", len(desc))

        # opcional: normalizar para uma linha (facilita abrir no Excel)
        desc = desc.replace("\r\n", "\n").strip()

        rows_buffer.append({
            "image_filename": fn,
            "llava_desc": desc,
            "seconds": round(time.time() - t0, 2),
            "status": "ok"
        })

    except Exception as e:
        rows_buffer.append({
            "image_filename": fn,
            "llava_desc": "",
            "seconds": round(time.time() - t0, 2),
            "status": f"error: {type(e).__name__}: {e}"
        })

    if len(rows_buffer) >= flush_every:
        df_part = pd.DataFrame(rows_buffer)
        header = not os.path.exists(out_csv)
        df_part.to_csv(out_csv, mode="a", header=header, index=False, encoding="utf-8")
        rows_buffer = []

# flush final
if rows_buffer:
    df_part = pd.DataFrame(rows_buffer)
    header = not os.path.exists(out_csv)
    df_part.to_csv(out_csv, mode="a", header=header, index=False, encoding="utf-8")

print("saved:", out_csv)


total images: 1 already done (ok): 10 to do: 0


0it [00:00, ?it/s]

saved: ./out/llava_captions.csv


In [14]:
from PIL import Image

test_path = "./images/sample/A_Adi Michael_Acquired taste_(2).jpg"  # troque pelo nome real
img = Image.open(test_path).convert("RGB")
img = resize_for_vlm(img, max_side=768)

desc = llava_describe(img, max_new_tokens=80)
print(desc)


KeyboardInterrupt: 

KeyboardInterrupt: 